<a href="https://colab.research.google.com/github/IvanMorsin/Forecasting-electrical-power-in-multi-storey-residential-buildings/blob/main/Notebook_17_new_building.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))


def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def load_data(fact_path, forecast_path):
    df_fact = pd.read_csv(fact_path)
    df_forecast = pd.read_csv(forecast_path)

    df_fact["timestamp"] = pd.to_datetime(df_fact["timestamp"])
    df_forecast["timestamp"] = pd.to_datetime(df_forecast["timestamp"])

    df_fact = df_fact.sort_values("timestamp")
    df_forecast = df_forecast.sort_values("timestamp")

    df = pd.merge_asof(
        df_fact,
        df_forecast,
        on="timestamp",
        direction="nearest",
        tolerance=pd.Timedelta("1min")
    )

    df = df.dropna(subset=["power_forecast"])

    return df
def plot_comparison(df, title):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df["timestamp"],
        y=df["power"],
        mode="lines",
        name="Факт",
        line=dict(shape="hv")
    ))

    fig.add_trace(go.Scatter(
        x=df["timestamp"],
        y=df["power_forecast"],
        mode="lines",
        name="Прогноз",
        line=dict(shape="hv")
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Время",
        yaxis_title="Power",
        template="plotly_white"
    )

    fig.show()

datasets = {
    "1m": ("fact_1m.csv", "forecast_1m.csv"),
    "14d": ("fact_14d.csv", "forecast_14d.csv"),
    "7d": ("fact_7d.csv", "forecast_7d.csv"),
    "24h": ("fact_24h.csv", "forecast_24h-2.csv"),
}

results = []

for name, (fact_file, forecast_file) in datasets.items():
    print(f"Dataset: {name}")

    df = load_data(fact_file, forecast_file)

    if len(df) == 0:
        continue

    y_true = df["power"]
    y_pred = df["power_forecast"]

    mae_val = mae(y_true, y_pred)
    mape_val = mape(y_true, y_pred)

    print(f"MAE: {mae_val:.4f}")
    print(f"MAPE: {mape_val:.2f}%")
    print(f"Точек: {len(df)}")

    results.append({
        "dataset": name,
        "MAE": mae_val,
        "MAPE": mape_val,
        "n_points": len(df)
    })

    plot_comparison(df, f"{name}: факт vs прогноз")

results_df = pd.DataFrame(results)

print("\Итог")
print(results_df)

Dataset: 1m
MAE: 7.7266
MAPE: 10.36%
Точек: 1488


Dataset: 14d
MAE: 6.8826
MAPE: 10.54%
Точек: 672


Dataset: 7d
MAE: 7.1281
MAPE: 11.47%
Точек: 336


Dataset: 24h
MAE: 4.8496
MAPE: 8.65%
Точек: 48


\Итог
  dataset       MAE       MAPE  n_points
0      1m  7.726647  10.361987      1488
1     14d  6.882649  10.539142       672
2      7d  7.128125  11.474041       336
3     24h  4.849583   8.646277        48
